# M4 — Versions A and B + the test harness

**Goal of this step:** build the first two versions of the system and a "harness" that grades any version on
the benchmark, automatically and the same way every time.

**The two versions** differ in only one thing: **which table cards the AI gets to read** before writing SQL.
| Version | What the AI sees | Analogy |
|---|---|---|
| **A. No search** | all 28 table cards | handing someone the whole catalog |
| **B. Basic search** | only the 5 cards Qdrant finds closest to the question (M2) | handing them the 5 most relevant catalog cards |

**The harness** is like an exam machine: for each question it runs a version, runs the SQL **safely**, compares the
result with the correct answer, and saves everything.

**Files in this step**
| File | What it does |
|---|---|
| `mcp_server/safety.py` | only one SELECT allowed (checked by a real SQL parser), read-only, 1,000 rows, 20 s limit |
| `agent/llm.py` | one function, `ask()`, that calls the OpenAI model and measures tokens and time |
| `agent/pipelines.py` | the prompt, `version_a()`, `version_b()` |
| `harness/metrics.py` | is the answer right? were the right tables used? |
| `harness/run.py` | runs a version on the questions → `results/<run_id>/` |
| `configs/A.yaml`, `configs/B.yaml` | the settings for each version (model, k, prices) |

**Done when:** the harness prints accuracy for A and B on the practice questions ✅

In [1]:
import sys, json, glob
for folder in ["../agent", "../catalog", "../harness", "../mcp_server"]:
    sys.path.append(folder)

## 1. Safety first
Whatever SQL the AI writes goes through `check_sql()` first. Anything that isn't a single SELECT is blocked.
On top of that, the database is opened **read-only**, so even a missed case couldn't change data.

In [2]:
from safety import check_sql
for sql in ["SELECT clicks FROM agg_campaign_daily", "DROP TABLE dim_ad", "DELETE FROM dim_ad",
            "SELECT 1; DROP TABLE dim_ad", "COPY dim_ad TO 'stolen.csv'"]:
    print(f"{sql:40s} -> {check_sql(sql) or 'allowed'}")

SELECT clicks FROM agg_campaign_daily    -> allowed
DROP TABLE dim_ad                        -> only SELECT queries are allowed
DELETE FROM dim_ad                       -> only SELECT queries are allowed
SELECT 1; DROP TABLE dim_ad              -> exactly one SQL statement is allowed
COPY dim_ad TO 'stolen.csv'              -> only SELECT queries are allowed


## 2. What the AI is told
The **system prompt** gives the fixed "now" and the rules. Then come the table cards and the question.

In [3]:
from pipelines import SYSTEM_PROMPT, CARDS, card_for_prompt
print(SYSTEM_PROMPT)
print("---- one card as the AI sees it ----")
print(card_for_prompt(CARDS["agg_campaign_daily_valid"]))

/Users/anshikapuneet/Documents/Claude_history/capstone_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You write DuckDB SQL for an ads data warehouse (Taobao ads, May 2017).

Facts about time:
- "Now" is 2017-05-13 23:59 (China time). Today = 2017-05-13. Yesterday = 2017-05-12.
- The data covers 2017-05-06 to 2017-05-13.

Rules:
- Use ONLY the tables and columns listed below.
- Write exactly ONE read-only SELECT query.
- Return only the columns needed to answer the question.
- CTR is a fraction (clicks / impressions), not a percentage.
- Reply with the SQL inside a ```sql code block and nothing else.

---- one card as the AI sees it ----
### agg_campaign_daily_valid
Final daily impressions, clicks, CTR and estimated value per campaign with suspicious / invalid / bot traffic removed.
One row = one row per campaign per day. Freshness: final. Traffic: valid.
Columns: event_date DATE, campaign_id BIGINT, advertiser_id BIGINT, impressions BIGINT, clicks HUGEINT, ctr DOUBLE, est_value DOUBLE
Dates: 2017-05-06 to 2017-05-12
Joins: campaign_id -> dim_campaign, advertiser_id -> dim_advertiser


## 3. One question through A and through B
This makes 2 real API calls (costs well under one cent).

In [4]:
from pipelines import version_a, version_b
import yaml
config_a = yaml.safe_load(open("../configs/A.yaml"))
config_b = yaml.safe_load(open("../configs/B.yaml"))

question = "How many unique male users saw campaign 258678 on 9 May 2017?"
for name, fn, cfg in [("A", version_a, config_a), ("B", version_b, config_b)]:
    out = fn(question, cfg)
    print(f"=== Version {name}: saw {len(out['tables_shown'])} cards, {out['tokens_in']} tokens in")
    if name == "B":
        print("cards shown:", out["tables_shown"])
    print(out["sql"], "\n")

=== Version A: saw 28 cards, 3212 tokens in
SELECT COUNT(DISTINCT f.user_id) AS unique_male_users
FROM fact_ad_events f
JOIN dim_user u ON f.user_id = u.user_id
WHERE f.campaign_id = 258678
  AND f.event_date = DATE '2017-05-09'
  AND u.gender = 'male' 

=== Version B: saw 5 cards, 702 tokens in
cards shown: ['agg_campaign_daily_by_audience', 'reach_campaign_total', 'live_reach_campaign_last6h', 'reach_campaign_daily', 'dim_user']
SELECT
  CAST(ROUND(
    rc.unique_users
    * (COALESCE(male_impr,0) / NULLIF(total_impr,0))
  ) AS BIGINT) AS unique_male_users
FROM
  reach_campaign_daily rc
LEFT JOIN (
  SELECT
    event_date,
    campaign_id,
    SUM(impressions) FILTER (WHERE gender = 'male') AS male_impr,
    SUM(impressions) AS total_impr
  FROM agg_campaign_daily_by_audience
  WHERE event_date = DATE '2017-05-09' AND campaign_id = 258678
  GROUP BY event_date, campaign_id
) a ON rc.event_date = a.event_date AND rc.campaign_id = a.campaign_id
WHERE rc.event_date = DATE '2017-05-09' A

Look at what B was shown. To count **male** users it needs `dim_user` (where gender lives) joined to
`fact_ad_events`. If search didn't bring `dim_user` into the top 5, B **can't** write the right query.

## 4. How answers are graded
`results_match(gold, predicted)`:
- row order doesn't matter
- numbers are compared after rounding to 4 decimals
- **extra columns are OK** (returning clicks *and* CTR when only CTR was asked)
- every gold column must be there, and the rows must line up

In [5]:
from metrics import results_match
print("extra column is fine      :", results_match([[0.0584]], [[0.05839416, 7, 120]]))
print("percent instead of fraction:", results_match([[0.0584]], [[5.84]]))
print("values swapped between rows:", results_match([["female", 0.1], ["male", 0.2]], [["male", 0.1], ["female", 0.2]]))

extra column is fine      : True
percent instead of fraction: False
values swapped between rows: False


**Self-test of the harness:** if the "prediction" is the gold SQL itself, every question must come out right.
If not, the grader has a bug.

In [6]:
import duckdb
from safety import run_sql

con = duckdb.connect("../data/ads.duckdb", read_only=True)
questions = [json.loads(l) for l in open("../benchmark/questions.jsonl")]
gold = {json.loads(l)["id"]: json.loads(l)["rows"] for l in open("../benchmark/gold_results.jsonl")}
right = 0
for q in questions:
    rows, cols, err = run_sql(con, open("../benchmark/" + q["gold_sql"]).read())
    right += (not err) and results_match(gold[q["id"]], rows)
con.close()
print(f"gold SQL graded as correct: {right} / {len(questions)}")

gold SQL graded as correct: 120 / 120


## 5. The practice results
Both versions were run on the **40 practice questions** from the terminal:
```
.venv/bin/python harness/run.py configs/A.yaml --split practice
.venv/bin/python harness/run.py configs/B.yaml --split practice
```
Here are the saved scores:

In [7]:
def latest(version):
    folder = sorted(glob.glob(f"../results/*_{version}_practice"))[-1]
    return folder, json.load(open(folder + "/metrics.json"))

keys = ["accuracy", "accuracy_easy", "accuracy_medium", "accuracy_tricky", "table_f1",
        "gold_tables_shown", "sql_errors", "avg_seconds", "avg_tokens_in", "avg_cost_usd"]
(fa, ma), (fb, mb) = latest("A"), latest("B")
print(f"{'metric':20s} {'A (all cards)':>15s} {'B (top 5)':>12s}")
for k in keys:
    print(f"{k:20s} {ma[k]:>15} {mb[k]:>12}")

metric                 A (all cards)    B (top 5)
accuracy                       0.875          0.7
accuracy_easy                    1.0          1.0
accuracy_medium               0.8462       0.3077
accuracy_tricky               0.7857       0.7857
table_f1                      0.9542       0.7208
gold_tables_shown                1.0         0.65
sql_errors                         0            0
avg_seconds                   3.7472       4.2538
avg_tokens_in                 3208.8      712.825
avg_cost_usd                  0.0012       0.0007


**How to read this**
- `accuracy` = share of questions with the right answer (the main score)
- `table_f1` = how well the tables used match the correct tables (1.0 = exactly right)
- `gold_tables_shown` = how often **all** the needed tables were among the cards the AI saw.
  For B this is the search's *recall@5*.

## 6. Look at the mistakes

In [8]:
def mistakes(folder):
    for line in open(folder + "/answers.jsonl"):
        r = json.loads(line)
        if not r["correct"]:
            print(f"{r['id']} {r['tier']:6s} {r['question'][:75]}")
            print(f"    needed tables shown: {r['gold_tables_shown']}   used: {r['tables_used']}   gold: {r['gold_tables']}")

print("=== Version A mistakes"); mistakes(fa)
print("\n=== Version B mistakes"); mistakes(fb)

=== Version A mistakes
q057 medium Which brand in category 4357 had the most clicks yesterday?
    needed tables shown: True   used: ['agg_brand_daily', 'dim_ad']   gold: ['agg_ad_daily', 'dim_ad']
q060 medium Top brand by clicks yesterday within category 6142?
    needed tables shown: True   used: ['agg_ad_daily', 'dim_ad']   gold: ['agg_ad_daily', 'dim_ad']
q101 tricky How many clicks did campaign 319897 get in the week starting Monday 8 May?
    needed tables shown: True   used: ['agg_campaign_weekly']   gold: ['agg_campaign_weekly']
q102 tricky Weekly clicks for campaign 133088, week of 8 May?
    needed tables shown: True   used: ['agg_campaign_weekly']   gold: ['agg_campaign_weekly']
q112 tricky Campaign 236820 CTR by gender on 7 May 2017?
    needed tables shown: True   used: ['agg_campaign_daily_by_audience']   gold: ['agg_campaign_daily_by_audience']

=== Version B mistakes
q047 medium What was the CTR among heavy shoppers (shopping depth 3) for campaign 12519
    needed table

## 7. What this tells us
- **A beats B at 28 tables.** The design doc predicted this risk ("Version A wins at 28 tables"), and it's a real
  finding. With only 28 cards (about 3,200 tokens), the AI can read everything, so search only takes information away.
- **B's mistakes are almost all medium (join) questions where search didn't show a needed table**
  (`dim_user`, `dim_ad`). Search finds the *main* table but not the *connecting* table. That's exactly what
  version **D (the Neo4j table map, M5)** is meant to fix: follow the map from the found table to its join partners.
- **Both make the same "look-alike" mistakes**, e.g. the audience table has one row per gender **per age level**,
  so "CTR by gender" must add the rows up. Version **E (agent, M7)** can look at sample rows and retry.
- **The bigger-catalog test (M8)** checks whether A's lead disappears at 100–250 tables, where showing everything
  stops working.
- **One scoring note (T6, weekly clicks):** both versions got the number right but left out `days_in_week`, the
  warning that the week has only 5 final days. The gold answer includes that warning on purpose (it's the trap),
  and the benchmark is frozen, so these count as wrong. This is recorded in the README.

These are **practice** numbers, used while building. Final scores (test questions, 3 runs each, confidence
intervals) come in M9.

**Next step (M5):** the Neo4j table map + version D.